<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/solutions/09_generative_solutions.ipynb)

# 09 · 生成模型与材料逆向设计 — 课堂练习参考解答

本 Notebook 汇总了第 09 讲三个主讲 Notebook 的全部课堂练习参考实现与结果分析：

- **`09a_vae_selfies.ipynb`**：
  - 练习 `09a-1`：重构精度与潜空间规整度的权衡（$\beta = 0.05$ vs. $\beta = 0.5$）；
  - 练习 `09a-2`：球面线性插值（Slerp）修正范数塌陷与潜空间岭回归梯度外推实验；
  - 练习 `09a-3`：“训练集照抄基线” vs. VAE 先验采样的生成指标盲区诊断。
- **`09b_goal_directed_generation.ipynb`**：
  - 练习 `09b-1`：精英微调子集比例的权衡（Top 1% vs. 10% vs. 30%）；
  - 练习 `09b-2`：无约束 vs. 适用域约束的片段爬山搜索与悲观置信下界（LCB）评估；
  - 练习 `09b-3`：树模型外推天花板验证（仅用 `PCE < 6%` 训练随机森林预测 `PCE > 8%` 顶尖分子）。
- **`09c_microscopy_diffusion_sr.ipynb`**：
  - 练习 `09c-1`：基于 Sobel 算子的一阶相界梯度损失评估；
  - 练习 `09c-2`：前向扩散信噪比（SNR）衰减与终端高斯分布无先验空洞验证；
  - 练习 `09c-3`：条件扩散微结构生成的目标体积分数 vs. 实测体积分数体视学校准。

In [ ]:
import os
import re
import math
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
import selfies as sf

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

RDLogger.DisableLog("rdApp.*")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "../data" if os.path.exists("../data/cep.csv") else "course/09 - Generative Models/data"
print(f"Using device: {device}, DATA_DIR: {DATA_DIR}")

---
## 第一部分：`09a` 课堂练习解答（SELFIES 变分自编码器与潜空间几何）

In [ ]:
# 加载 CEP 数据集、SELFIES 词表与预训练 MolecularVAE
df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
selfies_list = [sf.encoder(s) for s in df["smiles"]]
token_lists = [list(sf.split_selfies(s)) for s in selfies_list]
alphabet = ["[nop]"] + sorted({t for ts in token_lists for t in ts})
stoi = {t: i for i, t in enumerate(alphabet)}
max_len = max(len(ts) for ts in token_lists)

indices = torch.tensor([[stoi[t] for t in ts] + [0] * (max_len - len(ts)) for ts in token_lists], dtype=torch.long)
X = F.one_hot(indices, len(alphabet)).float().reshape(len(indices), -1)

perm = np.random.RandomState(0).permutation(len(X))
test_idx, train_idx = perm[:3000], perm[3000:]
LATENT_DIM = 128
INPUT_DIM = X.shape[1]

class MolecularVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, input_dim))

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z), mu, logvar

def decode_latent(z, model):
    model.eval()
    with torch.no_grad():
        logits = model.decoder(z)
    token_ids = logits.reshape(len(z), max_len, len(alphabet)).argmax(dim=-1)
    out = []
    for row in token_ids.tolist():
        s = "".join(alphabet[i] for i in row if i != 0)
        out.append(sf.decoder(s))
    return out

def canonical(smiles):
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None and mol.GetNumAtoms() > 0 else None

vae = MolecularVAE(INPUT_DIM, LATENT_DIM)
ckpt_vae = torch.load(os.path.join(DATA_DIR, "cep_vae.pt"), map_location="cpu")
vae.load_state_dict(ckpt_vae["model_state_dict"])
vae.eval()
with torch.no_grad():
    mu_test, _ = vae.encode(X[test_idx])
print("Loaded cep_vae.pt and encoded test set.")

### 练习 09a-1：验证重构精度与潜空间规整度的权衡（$\beta = 0.05$ vs. $\beta = 0.5$）

In [ ]:
def compare_beta_vae(beta_values=(0.05, 0.5), n_epochs=15):
    sub_train = train_idx[:5000]
    sub_test = test_idx[:500]
    orig_sub = [canonical(s) for s in df["smiles"].values[sub_test]]
    rows = []

    for beta_target in beta_values:
        torch.manual_seed(0)
        m_vae = MolecularVAE(INPUT_DIM, LATENT_DIM).to(device)
        opt = torch.optim.Adam(m_vae.parameters(), lr=1e-3)
        for ep in range(n_epochs):
            order = np.random.permutation(sub_train)
            rec_sum = kl_sum = 0.0
            b_val = beta_target * min(1.0, (ep + 1) / 5.0)
            for i in range(0, len(order), 256):
                xb = X[order[i : i + 256]].to(device)
                tb = indices[order[i : i + 256]].to(device)
                out, mu, logvar = m_vae(xb)
                rec = F.cross_entropy(out.reshape(-1, len(alphabet)), tb.reshape(-1), reduction="sum") / len(xb)
                kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / len(xb)
                loss = rec + b_val * kl
                opt.zero_grad()
                loss.backward()
                opt.step()
                rec_sum += rec.item()
                kl_sum += kl.item()
        m_vae.cpu().eval()
        with torch.no_grad():
            mu_sub, _ = m_vae.encode(X[sub_test])
        recon_sub = [canonical(s) for s in decode_latent(mu_sub, m_vae)]
        exact_rate = np.mean([a is not None and a == b for a, b in zip(recon_sub, orig_sub)])
        n_b = len(sub_train) / 256.0
        rows.append({
            "beta": beta_target,
            "Final Recon Loss": round(rec_sum / n_b, 2),
            "Final KL": round(kl_sum / n_b, 2),
            "Exact Match (%)": round(100.0 * exact_rate, 2),
            "Mean Std of mu": round(float(mu_sub.std(0).mean().item()), 3),
        })
    return pd.DataFrame(rows)

print(compare_beta_vae().to_string(index=False))

**结论**：当 $\beta$ 从 `0.05` 增大到 `0.5` 时，KL 散度被大幅压低，编码均值 $\boldsymbol{\mu}$ 的标准差显著收缩（部分潜维度发生后验坍缩，放弃编码结构信息），代价是重构损失翻倍、完全还原率大幅下跌。

### 练习 09a-2：(a) 球面线性插值（Slerp）修正范数塌陷 与 (b) 潜空间梯度外推

In [ ]:
def slerp(a, b, t):
    """球面线性插值：保持潜向量模长平滑过渡，避免掉入高维球壳内部空腔。"""
    a_n = a / a.norm()
    b_n = b / b.norm()
    omega = torch.acos((a_n * b_n).sum().clamp(-1.0 + 1e-6, 1.0 - 1e-6))
    r = (1.0 - t) * a.norm() + t * b.norm()
    return r * (torch.sin((1.0 - t) * omega) * a_n + torch.sin(t * omega) * b_n) / torch.sin(omega)

z_a, z_b = mu_test[0], mu_test[7]
ts = torch.linspace(0, 1, 8)
z_lerp = torch.stack([(1.0 - t) * z_a + t * z_b for t in ts])
z_slerp = torch.stack([slerp(z_a, z_b, t) for t in ts])

print("t     | ||z_lerp||_2 | ||z_slerp||_2")
for t_val, nl, ns in zip(ts.tolist(), z_lerp.norm(dim=1).tolist(), z_slerp.norm(dim=1).tolist()):
    print(f"{t_val:.2f}  |    {nl:5.2f}     |     {ns:5.2f}")

In [ ]:
# (b) 潜空间 Ridge 回归与梯度外推实验
pce_test = df["PCE"].values[test_idx]
Z_tr, Z_te, y_ztr, y_zte = train_test_split(mu_test.numpy(), pce_test, test_size=0.3, random_state=42)
ridge = Ridge(alpha=1.0).fit(Z_tr, y_ztr)
print(f"Latent Ridge Regression R^2 on held-out test set: {ridge.score(Z_te, y_zte):.3f}")

best_i = int(np.argmax(pce_test))
z0 = mu_test[best_i]
w_dir = torch.tensor(ridge.coef_, dtype=torch.float32)
w_dir = w_dir / w_dir.norm()

extrap_rows = []
for step_alpha in [0.0, 2.0, 5.0, 10.0, 20.0]:
    z_step = (z0 + step_alpha * w_dir).unsqueeze(0)
    pred_pce = float(ridge.predict(z_step.numpy())[0])
    dec_smi = canonical(decode_latent(z_step, vae)[0])
    mol_step = Chem.MolFromSmiles(dec_smi) if dec_smi else None
    extrap_rows.append({
        "Step Alpha": step_alpha,
        " ||z||_2": round(float(z_step.norm().item()), 2),
        "Ridge Predicted PCE (%)": round(pred_pce, 2),
        "Aromatic Rings": Descriptors.NumAromaticRings(mol_step) if mol_step else 0,
        "Decoded SMILES": dec_smi[:45] if dec_smi else "Invalid",
    })

pd.DataFrame(extrap_rows)

### 练习 09a-3：“训练集照抄基线” vs. VAE 先验采样的生成指标盲区诊断

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

# 1. 照抄训练集 1000 个样本
mem_smiles = [Chem.CanonSmiles(s) for s in df["smiles"].iloc[train_idx[:1000]]]
mem_mols = [Chem.MolFromSmiles(s) for s in mem_smiles]

# 2. VAE 先验采样 1000 个样本
torch.manual_seed(0)
z_p = torch.randn(1000, LATENT_DIM)
prior_raw = [canonical(s) for s in decode_latent(z_p, vae)]
prior_ok = [c for c in prior_raw if c]
prior_mols = [Chem.MolFromSmiles(c) for c in prior_ok]

diag_df = pd.DataFrame([
    {
        "Generator": "Training Set Copy (Memorization)",
        "Validity": 1.000,
        "Uniqueness": round(len(set(mem_smiles)) / len(mem_smiles), 3),
        "Novelty": round(len(set(mem_smiles) - canonical_train) / len(set(mem_smiles)), 3),
        "Mean Heavy Atoms": round(float(np.mean([m.GetNumHeavyAtoms() for m in mem_mols])), 1),
        "Mean Aromatic Rings": round(float(np.mean([Descriptors.NumAromaticRings(m) for m in mem_mols])), 2),
        "Mean FractionCSP3": round(float(np.mean([Descriptors.FractionCSP3(m) for m in mem_mols])), 3),
    },
    {
        "Generator": "VAE Prior Sampling z ~ N(0,I)",
        "Validity": round(len(prior_ok) / 1000.0, 3),
        "Uniqueness": round(len(set(prior_ok)) / len(prior_ok), 3),
        "Novelty": round(len(set(prior_ok) - canonical_train) / len(set(prior_ok)), 3),
        "Mean Heavy Atoms": round(float(np.mean([m.GetNumHeavyAtoms() for m in prior_mols])), 1),
        "Mean Aromatic Rings": round(float(np.mean([Descriptors.NumAromaticRings(m) for m in prior_mols])), 2),
        "Mean FractionCSP3": round(float(np.mean([Descriptors.FractionCSP3(m) for m in prior_mols])), 3),
    },
])
diag_df

---
## 第二部分：`09b` 课堂练习解答（目标导向微调与代理模型欺骗防御）

In [ ]:
# 准备 09b 的随机森林代理模型与 SmilesRNN 先验模型
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = np.array([fp_gen.GetFingerprintAsNumPy(m) for m in mols], dtype=np.uint8)
y = df["PCE"].values

tr_i, te_i = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(fingerprints[tr_i], y[tr_i])

TOKEN_PATTERN = re.compile(r"(\[[^\]]+\]|Br|Cl|[BCNOPSFIbcnops]|\(|\)|\.|=|#|-|\+|\\|/|:|%\d{2}|\d)")
def tokenize(smiles):
    return TOKEN_PATTERN.findall(smiles)

tok_lists_smi = [tokenize(s) for s in df["smiles"]]
vocab_smi = ["<pad>", "^", "$"] + sorted({t for ts in tok_lists_smi for t in ts})
stoi_smi = {t: i for i, t in enumerate(vocab_smi)}
max_len_smi = max(len(ts) for ts in tok_lists_smi) + 2

data_smi = torch.zeros(len(tok_lists_smi), max_len_smi, dtype=torch.long)
for i, tokens in enumerate(tok_lists_smi):
    ids = [stoi_smi["^"]] + [stoi_smi[t] for t in tokens] + [stoi_smi["$"]]
    data_smi[i, : len(ids)] = torch.tensor(ids)

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, n_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        out, state = self.lstm(self.embedding(x), state)
        return self.linear(out), state

@torch.no_grad()
def sample_smi(model, n=600, temperature=1.0):
    model.eval()
    x = torch.full((n, 1), stoi_smi["^"], dtype=torch.long)
    state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_len_smi):
        logits, state = model(x, state)
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)
        out = torch.cat([out, x], dim=1)
        finished |= (x[:, 0] == stoi_smi["$"])
        if finished.all():
            break
    res = []
    for row in out.tolist():
        toks = []
        for t in row:
            if t == stoi_smi["$"]:
                break
            toks.append(vocab_smi[t])
        res.append("".join(toks))
    return res

prior_model = SmilesRNN(len(vocab_smi))
raw_ckpt = torch.load(os.path.join(DATA_DIR, "cep_rnn.pt"), map_location="cpu")
prior_model.load_state_dict(raw_ckpt["model_state_dict"] if isinstance(raw_ckpt, dict) and "model_state_dict" in raw_ckpt else raw_ckpt)
prior_model.eval()
print("Ready for 09b solutions.")

### 练习 09b-1：精英子集比例的权衡实验（Top 1% vs. 10% vs. 30%）

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

def finetune_and_eval(fraction, n_epochs=4, lr=1e-4, n_eval=600):
    model = copy.deepcopy(prior_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n_top = int(len(df) * fraction)
    subset = data_smi[np.argsort(-y)[:n_top]]
    model.train()
    for _ in range(n_epochs):
        order = np.random.permutation(len(subset))
        for i in range(0, len(order), 128):
            batch = subset[order[i : i + 128]]
            logits, _ = model(batch[:, :-1])
            loss = loss_fn(logits.reshape(-1, len(vocab_smi)), batch[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    samples = sample_smi(model, n=n_eval)
    valid = []
    for s in samples:
        m = Chem.MolFromSmiles(s) if s else None
        if m is not None and m.GetNumAtoms() > 0:
            valid.append(Chem.MolToSmiles(m, canonical=True))
    uniq = sorted(set(valid))
    fps = np.array([fp_gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(c)) for c in uniq], dtype=np.uint8)
    preds = rf.predict(fps)
    return {
        "Elite Subset": f"Top {int(fraction*100)}% ({n_top} mols)",
        "Validity": round(len(valid) / n_eval, 3),
        "Uniqueness": round(len(uniq) / max(len(valid), 1), 3),
        "Novelty": round(len(set(uniq) - canonical_train) / max(len(uniq), 1), 3),
        "Mean Pred PCE (%)": round(float(preds.mean()), 2),
    }

rows_09b1 = [finetune_and_eval(f) for f in (0.01, 0.10, 0.30)]
pd.DataFrame(rows_09b1)

### 练习 09b-2：无约束 vs. 适用域约束的片段爬山搜索（Hill Climbing）

In [ ]:
FRAGMENTS = [
    "c1ccsc1", "c1ccoc1", "c1cc[nH]c1", "c1ccccc1",
    "c1cc[se]c1", "c1ccc2nsnc2c1", "c1ccc2[nH]ccc2c1",
]
ref_idx = np.random.RandomState(0).choice(len(df), 2000, replace=False)
ref_fps = [fp_gen.GetFingerprint(mols[i]) for i in ref_idx]

def tree_predictions(mol):
    x = np.array([fp_gen.GetFingerprintAsNumPy(mol)], dtype=np.uint8)
    return np.array([t.predict(x)[0] for t in rf.estimators_])

def sim_to_train(mol):
    return max(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mol), ref_fps))

def hill_climb(constrained=False, n_steps=120, seed=0):
    rng = random.Random(seed)
    chain = ["c1ccsc1", "c1ccc2nsnc2c1", "c1ccsc1"]
    best_mol = Chem.MolFromSmiles("-".join(chain))

    def score_fn(mol):
        preds = tree_predictions(mol)
        return float(preds.mean() - 2.0 * preds.std()) if constrained else float(preds.mean())

    best_score = score_fn(best_mol)
    for _ in range(n_steps):
        cand = chain.copy()
        op = rng.choice(["insert", "delete", "replace"])
        if op == "insert" and len(cand) < 10:
            cand.insert(rng.randrange(len(cand) + 1), rng.choice(FRAGMENTS))
        elif op == "delete" and len(cand) > 2:
            cand.pop(rng.randrange(len(cand)))
        else:
            cand[rng.randrange(len(cand))] = rng.choice(FRAGMENTS)

        mol = Chem.MolFromSmiles("-".join(cand))
        if mol is None:
            continue
        if constrained and sim_to_train(mol) < 0.40:
            continue
        s = score_fn(mol)
        if s > best_score:
            chain, best_mol, best_score = cand, mol, s
    return chain, best_mol

hc_rows = []
for flag in (False, True):
    ch, m = hill_climb(constrained=flag)
    tp = tree_predictions(m)
    hc_rows.append({
        "Strategy": "Constrained (Sim>=0.4 + LCB)" if flag else "Unconstrained (Mean Only)",
        "Pred Mean PCE (%)": round(float(tp.mean()), 2),
        "Pessimistic LCB (mu - 2*std)": round(float(tp.mean() - 2.0 * tp.std()), 2),
        "Tree Spread Std (%)": round(float(tp.std()), 2),
        "Max Tanimoto Sim": round(sim_to_train(m), 3),
        "Heavy Atoms": m.GetNumHeavyAtoms(),
        "Fragments": len(ch),
    })
pd.DataFrame(hc_rows)

### 练习 09b-3：检验树模型的外推天花板（仅用 `PCE < 6%` 训练随机森林预测 `PCE > 8%`）

In [ ]:
low_mask = y < 6.0
high_mask = y > 8.0
rf_low = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_low.fit(fingerprints[low_mask], y[low_mask])
pred_high = rf_low.predict(fingerprints[high_mask])

print(f"Trained on {low_mask.sum():,} molecules with PCE < 6.0%; tested on {high_mask.sum():,} molecules with PCE > 8.0%")
print(f"True Mean PCE      : {y[high_mask].mean():.2f}%")
print(f"Predicted Mean PCE : {pred_high.mean():.2f}% (Mean Underestimation = {y[high_mask].mean() - pred_high.mean():.2f}%)")
print(f"Fraction Predicted > 6.0% (Training Ceiling): {np.mean(pred_high > 6.0):.1%}")

---
## 第三部分：`09c` 课堂练习解答（显微图像超分与条件去噪扩散模型）

In [ ]:
# 加载 09c 的合成微结构与预训练权重
def generate_two_phase_microstructures(n_samples=800, img_size=32, seed=42):
    rng = np.random.RandomState(seed)
    hr_imgs = np.zeros((n_samples, 1, img_size, img_size), dtype=np.float32)
    phis = np.zeros(n_samples, dtype=np.float32)
    for i in range(n_samples):
        raw = rng.randn(img_size, img_size)
        sigma = rng.uniform(1.8, 2.6)
        field = gaussian_filter(raw, sigma=sigma, mode="wrap")
        field = (field - field.mean()) / (field.std() + 1e-8)
        target_phi = rng.uniform(0.25, 0.75)
        thresh = np.quantile(field, 1.0 - target_phi)
        phase = 0.5 * (1.0 + np.tanh((field - thresh) / 0.18))
        hr_imgs[i, 0] = phase.astype(np.float32)
        phis[i] = float((phase > 0.5).mean())
    return torch.tensor(hr_imgs), torch.tensor(phis)

def degrade_sem_images(hr_tensor, noise_std=0.05, seed=123):
    lr_clean = F.avg_pool2d(hr_tensor, kernel_size=2, stride=2)
    gen = torch.Generator().manual_seed(seed)
    noise = torch.randn(lr_clean.shape, generator=gen) * noise_std
    return (lr_clean + noise).clamp(0.0, 1.0)

hr_all, phi_all = generate_two_phase_microstructures(800, 32, seed=42)
lr_all = degrade_sem_images(hr_all, 0.05, seed=123)
test_hr, test_lr = hr_all[640:], lr_all[640:]

class MicrostructureSRNet(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, 1, 3, padding=1),
        )

    def forward(self, x_lr):
        x_up = F.interpolate(x_lr, scale_factor=2, mode="bicubic", align_corners=False)
        return (x_up + self.body(x_up)).clamp(0.0, 1.0)

ckpt_09c = torch.load(os.path.join(DATA_DIR, "micro_sr_ddpm.pt"), map_location=device)
sr_net = MicrostructureSRNet(32).to(device)
sr_net.load_state_dict(ckpt_09c["sr_state_dict"])
sr_net.eval();

### 练习 09c-1：构建 Sobel 边缘梯度损失（Edge Gradient Loss）评估相界清晰度

In [ ]:
def sobel_gradient_loss(pred, target):
    kx = torch.tensor([[[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]], device=pred.device).unsqueeze(0)
    ky = torch.tensor([[[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]]], device=pred.device).unsqueeze(0)
    gx_p, gy_p = F.conv2d(pred, kx, padding=1), F.conv2d(pred, ky, padding=1)
    gx_t, gy_t = F.conv2d(target, kx, padding=1), F.conv2d(target, ky, padding=1)
    return (F.l1_loss(gx_p, gx_t) + F.l1_loss(gy_p, gy_t)).item()

with torch.no_grad():
    bic_test = F.interpolate(test_lr, scale_factor=2, mode="bicubic", align_corners=False).clamp(0.0, 1.0)
    sr_test = sr_net(test_lr.to(device)).cpu()

print(f"Sobel Edge Gradient Error | Bicubic: {sobel_gradient_loss(bic_test, test_hr):.4f}")
print(f"Sobel Edge Gradient Error | SR-Net : {sobel_gradient_loss(sr_test, test_hr):.4f} (Sharp phase boundaries restored)")

### 练习 09c-2：验证前向扩散信噪比（SNR）衰减与终端先验对齐

In [ ]:
T_STEPS = ckpt_09c["T_STEPS"]
betas = ckpt_09c["betas"]
alphas = 1.0 - betas
alphas_bar = torch.cumprod(alphas, dim=0)

snr_db = 10.0 * torch.log10(alphas_bar / (1.0 - alphas_bar))
print(f"SNR at t=0  : {snr_db[0]:+6.2f} dB (Pure signal)")
print(f"SNR at t=19 : {snr_db[19]:+6.2f} dB")
print(f"SNR at t=39 : {snr_db[-1]:+6.2f} dB (Noise dominated)")

x0_16 = F.avg_pool2d(hr_all, 2, 2) * 2.0 - 1.0
torch.manual_seed(0)
ab_last = alphas_bar[-1]
x_T = torch.sqrt(ab_last) * x0_16 + torch.sqrt(1.0 - ab_last) * torch.randn_like(x0_16)
pure_gauss = torch.randn_like(x0_16)

print(f"Median ||x_T||_2 at final forward step : {x_T.flatten(1).norm(dim=1).median():.2f}")
print(f"Median ||eps||_2 of pure N(0,I) prior  : {pure_gauss.flatten(1).norm(dim=1).median():.2f} (Zero prior hole!)")

### 练习 09c-3：条件扩散微结构生成的体视学校准曲线（Target $\phi$ vs. Measured $\hat{\phi}$）

In [ ]:
class ConditionalMicroUNet(nn.Module):
    def __init__(self, base_ch=32, T=40):
        super().__init__()
        self.T = T
        self.cond_mlp = nn.Sequential(nn.Linear(2, base_ch), nn.SiLU(), nn.Linear(base_ch, base_ch))
        self.enc1 = nn.Sequential(nn.Conv2d(1, base_ch, 3, padding=1), nn.GroupNorm(4, base_ch), nn.SiLU())
        self.down = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)
        self.mid = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, padding=1), nn.GroupNorm(8, base_ch * 2), nn.SiLU(),
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, padding=1), nn.GroupNorm(8, base_ch * 2), nn.SiLU(),
        )
        self.up = nn.ConvTranspose2d(base_ch * 2, base_ch, 4, stride=2, padding=1)
        self.out_conv = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch, 3, padding=1), nn.GroupNorm(4, base_ch), nn.SiLU(),
            nn.Conv2d(base_ch, 1, 3, padding=1),
        )

    def forward(self, x_t, t, phi):
        cond = self.cond_mlp(torch.cat([t.float().unsqueeze(1) / float(self.T), phi.float().unsqueeze(1)], dim=1))[:, :, None, None]
        h1 = self.enc1(x_t) + cond
        h2 = F.silu(self.down(h1))
        h_up = self.up(self.mid(h2))
        return self.out_conv(torch.cat([h_up, h1], dim=1))

ddpm_net = ConditionalMicroUNet(32, T_STEPS).to(device)
ddpm_net.load_state_dict(ckpt_09c["ddpm_state_dict"])
ddpm_net.eval()

@torch.no_grad()
def sample_ddpm_batch(target_phis, seed=0):
    torch.manual_seed(seed)
    n = len(target_phis)
    phi_t = torch.tensor(target_phis, dtype=torch.float32, device=device)
    x = torch.randn(n, 1, 16, 16, device=device)
    for step in reversed(range(T_STEPS)):
        t_b = torch.full((n,), step, dtype=torch.long, device=device)
        eps_hat = ddpm_net(x, t_b, phi_t)
        a_t, ab_t, b_t = alphas[step].to(device), alphas_bar[step].to(device), betas[step].to(device)
        mean = (1.0 / torch.sqrt(a_t)) * (x - (b_t / torch.sqrt(1.0 - ab_t)) * eps_hat)
        x = mean + (torch.sqrt(b_t) * torch.randn_like(x) if step > 0 else 0.0)
    return ((x.clamp(-1.0, 1.0) + 1.0) * 0.5).cpu()

calib_rows = []
for phi_tar in [0.30, 0.45, 0.55, 0.70]:
    imgs = sample_ddpm_batch([phi_tar] * 20, seed=42)
    measured = (imgs > 0.5).float().mean(dim=(1, 2, 3)).numpy()
    calib_rows.append({
        "Target Volume Fraction (phi)": phi_tar,
        "Measured Mean phi_hat": round(float(measured.mean()), 3),
        "Measured Std": round(float(measured.std()), 3),
    })

pd.DataFrame(calib_rows)